# Qualitätsprüfung: IPA-Transkriptionen (3 Varianten im Vergleich)

Dieses Notebook plausibilisiert die bereinigten IPA-Transkriptionen aus `transcriptions_clean.csv` (nur Ostschweiz).

**Drei Varianten:**
- `ipa_reference` – manuell erstellte Referenztranskription (Ground Truth)
- `ipa_audio` – automatisch via neurlang/ipa-whisper-base (Audio → IPA)
- `ipa_swiss_whisper` – automatisch via Swiss Whisper (Audio → Text → IPA-ähnlich)

## Stufe 1: Daten laden & Sichtprüfung

10 zufällige Samples anschauen: Wie unterscheiden sich die drei Varianten?

In [ ]:
import unicodedata
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "Data"

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

IPA_COLS = ['ipa_reference', 'ipa_audio', 'ipa_swiss_whisper']
COL_LABELS = {
    'ipa_reference':     'Reference (Ground Truth)',
    'ipa_audio':         'ipa-whisper-base (Audio)',
    'ipa_swiss_whisper': 'Swiss Whisper',
}
COLORS = ['#4C72B0', '#DD8452', '#55A868']

df = pd.read_csv(DATA_DIR / "transcriptions_clean.csv")
for col in IPA_COLS:
    df[col] = df[col].fillna('')

print(f'Geladen: {len(df):,} Zeilen, Region: {df["dialect_region"].unique().tolist()}')
print(f'Spalten: {list(df.columns)}')

In [ ]:
# 5 zufällige Samples – alle drei Varianten nebeneinander
sample = df.sample(5, random_state=42)

for _, row in sample.iterrows():
    print(f"Satz:      {row['sentence']}")
    print(f"Reference: {row['ipa_reference']}")
    print(f"Audio:     {row['ipa_audio']}")
    print(f"SwissWh.:  {row['ipa_swiss_whisper']}")
    print()

### Was wir prüfen:
- Enthalten alle Varianten echte IPA-Zeichen (`ə`, `ʊ`, `ɪ`, `ː` etc.)?
- Gibt es leere oder sehr kurze Outputs?
- Wie stark weichen die automatischen Varianten von der Referenz ab?

In [ ]:
# Schnellcheck: Leere / kurze Outputs pro Variante
print(f'{"Variante":<25} {"Leer":>6} {"< 5 Zeichen":>12} {"Ø Länge":>10} {"Min":>6} {"Max":>6}')
print('-' * 65)
for col in IPA_COLS:
    s = df[col].str.len()
    print(f'{COL_LABELS[col]:<25} {(s == 0).sum():>6} {(s < 5).sum():>12} {s.mean():>10.1f} {s.min():>6} {s.max():>6}')

## Stufe 2: IPA-Zeichenanteil – Wie «IPA-artig» ist jede Variante?

Anteil echter IPA-Sonderzeichen (nicht im normalen Deutschen vorhanden) an der Gesamtlänge.

In [ ]:
# IPA-spezifische Zeichen (nicht im normalen Deutschen vorhanden)
IPA_SPECIAL = set('əɛɪʊɔœæɑɐʌɜɒðŋɡʃʒθβχɣɸɹɻɺɾʀʁʋʍɥɰɫɬɮɭɳɵɤɯʔːˈˌ')

def ipa_ratio(text):
    if not isinstance(text, str) or len(text) == 0:
        return 0.0
    return sum(1 for c in text if c in IPA_SPECIAL) / len(text)

for col in IPA_COLS:
    df[f'_ratio_{col}'] = df[col].apply(ipa_ratio)

print(f'{"Variante":<25} {"Ø IPA-Anteil":>13} {"Median":>8} {"Std":>8} {"Min":>8} {"Max":>8}')
print('-' * 72)
for col in IPA_COLS:
    s = df[f'_ratio_{col}']
    print(f'{COL_LABELS[col]:<25} {s.mean():>13.3f} {s.median():>8.3f} {s.std():>8.3f} {s.min():>8.3f} {s.max():>8.3f}')

print('\nErwartung: Reference und Audio > 0.30 | Swiss Whisper deutlich niedriger (Textbasis)')

In [ ]:
# Visualisierung: IPA-Anteil aller drei Varianten
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogramme überlagert
for col, color in zip(IPA_COLS, COLORS):
    axes[0].hist(df[f'_ratio_{col}'], bins=40, alpha=0.55,
                 color=color, label=COL_LABELS[col], edgecolor='none')
axes[0].axvline(0.10, color='red', linestyle='--', linewidth=1.2, label='Schwelle 10%')
axes[0].set_xlabel('Anteil IPA-Sonderzeichen')
axes[0].set_ylabel('Anzahl Samples')
axes[0].set_title('IPA-Zeichenanteil: Verteilung der drei Varianten')
axes[0].legend(fontsize=9)

# Boxplot nebeneinander
ratio_data = [df[f'_ratio_{col}'].values for col in IPA_COLS]
bp = axes[1].boxplot(ratio_data, patch_artist=True,
                     labels=[COL_LABELS[c] for c in IPA_COLS])
for patch, color in zip(bp['boxes'], COLORS):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
axes[1].set_ylabel('Anteil IPA-Sonderzeichen')
axes[1].set_title('IPA-Zeichenanteil: Varianten im Vergleich')
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()

## Stufe 3: Bekannte Schweizerdeutsche Laute – Häufigkeit pro Variante

Wie oft tauchen charakteristische Laute in den drei Varianten auf?

In [ ]:
LAUT_CHECKS = {
    '/x/ (ch-Laut)':       'x',
    '/ː/ (lange Vokale)':  'ː',
    '/ʃ/ (sch-Laut)':      'ʃ',
    '/ə/ (Schwa)':         'ə',
    '/ɪ/ (kurzes i)':      'ɪ',
    '/ʊ/ (kurzes u)':      'ʊ',
    '/r/-Varianten':       r'[rɾʀʁ]',
}

print(f'{"Laut":<22}', end='')
for col in IPA_COLS:
    print(f'{COL_LABELS[col]:>27}', end='')
print()
print('-' * 103)

for label, pattern in LAUT_CHECKS.items():
    print(f'{label:<22}', end='')
    for col in IPA_COLS:
        freq = df[col].str.contains(pattern, regex=('|' in pattern or '[' in pattern), na=False).mean()
        print(f'{freq:>27.1%}', end='')
    print()

In [ ]:
# Visualisierung: Lautfrequenz pro Variante
laut_labels = list(LAUT_CHECKS.keys())
x = np.arange(len(laut_labels))
width = 0.25

fig, ax = plt.subplots(figsize=(13, 5))
for i, (col, color) in enumerate(zip(IPA_COLS, COLORS)):
    freqs = [
        df[col].str.contains(pat, regex=('|' in pat or '[' in pat), na=False).mean()
        for pat in LAUT_CHECKS.values()
    ]
    ax.bar(x + i * width, freqs, width, label=COL_LABELS[col],
           color=color, alpha=0.8, edgecolor='white')

ax.set_xticks(x + width)
ax.set_xticklabels(laut_labels, rotation=25, ha='right')
ax.set_ylabel('Anteil Samples mit diesem Laut')
ax.set_title('Häufigkeit schweizerdeutscher IPA-Laute – drei Varianten im Vergleich')
ax.legend(fontsize=9)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

## Stufe 4: Direktvergleich der drei Varianten

Wie ähnlich sind sich `ipa_audio` und `ipa_swiss_whisper` zur Referenz?  
Metriken: Länge, Zeichenüberlappung (Jaccard auf Zeichenebene), Character Error Rate (CER).

In [ ]:
import Levenshtein  

def cer(ref, hyp):
    """Character Error Rate: Levenshtein-Distanz / Länge Referenz."""
    if len(ref) == 0:
        return float('nan')
    return Levenshtein.distance(ref, hyp) / len(ref)

def jaccard_chars(a, b):
    """Jaccard-Ähnlichkeit auf Zeichen-Multiset-Basis."""
    if not a and not b:
        return 1.0
    sa, sb = set(a), set(b)
    return len(sa & sb) / len(sa | sb)

# Metriken berechnen (vs. ipa_reference)
for col in ['ipa_audio', 'ipa_swiss_whisper']:
    df[f'_cer_{col}']     = df.apply(lambda r: cer(r['ipa_reference'], r[col]), axis=1)
    df[f'_jacc_{col}']    = df.apply(lambda r: jaccard_chars(r['ipa_reference'], r[col]), axis=1)
    df[f'_lenratio_{col}'] = df[col].str.len() / df['ipa_reference'].str.len().replace(0, float('nan'))

print('=== Character Error Rate vs. ipa_reference (niedriger = besser) ===')
for col in ['ipa_audio', 'ipa_swiss_whisper']:
    s = df[f'_cer_{col}'].dropna()
    print(f'  {COL_LABELS[col]:<30}  Ø={s.mean():.3f}  Median={s.median():.3f}  Std={s.std():.3f}')

print()
print('=== Jaccard-Ähnlichkeit vs. ipa_reference (höher = besser) ===')
for col in ['ipa_audio', 'ipa_swiss_whisper']:
    s = df[f'_jacc_{col}']
    print(f'  {COL_LABELS[col]:<30}  Ø={s.mean():.3f}  Median={s.median():.3f}  Std={s.std():.3f}')

print()
print('=== Längenverhältnis vs. ipa_reference (1.0 = gleich lang) ===')
for col in ['ipa_audio', 'ipa_swiss_whisper']:
    s = df[f'_lenratio_{col}'].dropna()
    print(f'  {COL_LABELS[col]:<30}  Ø={s.mean():.3f}  Median={s.median():.3f}  Std={s.std():.3f}')

In [ ]:
# Visualisierung: CER und Längenverhältnis
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

auto_cols = ['ipa_audio', 'ipa_swiss_whisper']
auto_labels = [COL_LABELS[c] for c in auto_cols]
auto_colors = COLORS[1:]

# CER Histogramme
for col, color, label in zip(auto_cols, auto_colors, auto_labels):
    axes[0].hist(df[f'_cer_{col}'].dropna(), bins=40, alpha=0.6,
                 color=color, label=label, edgecolor='none')
axes[0].set_xlabel('Character Error Rate')
axes[0].set_ylabel('Anzahl Samples')
axes[0].set_title('CER vs. ipa_reference')
axes[0].set_xlim(0, 1)
axes[0].legend(fontsize=9)

# Jaccard Histogramme
for col, color, label in zip(auto_cols, auto_colors, auto_labels):
    axes[1].hist(df[f'_jacc_{col}'], bins=40, alpha=0.6,
                 color=color, label=label, edgecolor='none')
axes[1].set_xlabel('Jaccard-Ähnlichkeit (Zeichenebene)')
axes[1].set_title('Jaccard vs. ipa_reference')
axes[1].legend(fontsize=9)

# Längenverhältnis Boxplot
len_data = [df[f'_lenratio_{col}'].dropna().values for col in auto_cols]
bp = axes[2].boxplot(len_data, patch_artist=True, labels=auto_labels)
for patch, color in zip(bp['boxes'], auto_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
axes[2].axhline(1.0, color='black', linestyle='--', linewidth=1, label='Referenzlänge')
axes[2].set_ylabel('Länge relativ zu ipa_reference')
axes[2].set_title('Längenverhältnis vs. ipa_reference')
axes[2].legend(fontsize=9)
axes[2].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

# 3 Beispiele mit hoher / niedriger CER
print('\n--- 3 Samples mit niedrigster CER (ipa_audio) ---')
for _, row in df.nsmallest(3, '_cer_ipa_audio').iterrows():
    print(f'  CER={row["_cer_ipa_audio"]:.3f} | Ref: {row["ipa_reference"]} | Audio: {row["ipa_audio"]}')

print('\n--- 3 Samples mit höchster CER (ipa_audio) ---')
for _, row in df.nlargest(3, '_cer_ipa_audio').iterrows():
    print(f'  CER={row["_cer_ipa_audio"]:.3f} | Ref: {row["ipa_reference"]} | Audio: {row["ipa_audio"]}')

In [ ]:
# CER-Histogramm: voller Bereich (inkl. Halluzinationen mit CER >> 1)
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

for col, color, label, row_axes in zip(
    ['ipa_audio', 'ipa_swiss_whisper'],
    COLORS[1:],
    auto_labels,
    [axes[0], axes[1]]
):
    cer_vals = df[f'_cer_{col}'].dropna()
    cer_min, cer_max = cer_vals.min(), cer_vals.max()

    # --- Linkes Panel: voller Bereich ---
    ax_full = row_axes[0]
    ax_full.hist(cer_vals, bins=100, color=color, alpha=0.75, edgecolor='none',
                 range=(cer_min, cer_max))
    for p, ls in [(50, '--'), (75, '-.'), (90, ':'), (95, '-')]:
        val = cer_vals.quantile(p / 100)
        ax_full.axvline(val, linestyle=ls, linewidth=1.2, color='black',
                        label=f'P{p} = {val:.2f}')
    ax_full.set_xlabel('Character Error Rate (CER)')
    ax_full.set_ylabel('Anzahl Samples')
    ax_full.set_title(f'{label}\nVoller Bereich: {cer_min:.3f} – {cer_max:.3f}')
    ax_full.set_xlim(cer_min, cer_max)
    ax_full.legend(fontsize=9)

    # --- Rechtes Panel: Zoom 0–2 (wo die meisten Samples liegen) ---
    ax_zoom = row_axes[1]
    ax_zoom.hist(cer_vals, bins=80, color=color, alpha=0.75, edgecolor='none',
                 range=(0, 2))
    for p, ls in [(50, '--'), (75, '-.'), (90, ':'), (95, '-')]:
        val = cer_vals.quantile(p / 100)
        if val <= 2:
            ax_zoom.axvline(val, linestyle=ls, linewidth=1.2, color='black',
                            label=f'P{p} = {val:.2f}')
    ax_zoom.axvline(1.0, color='red', linewidth=1.5, linestyle='-',
                    label='CER = 1.0 (Halluzinationsgrenze)')
    n_above = (cer_vals > 1.0).sum()
    ax_zoom.text(1.02, 0.95, f'{n_above} Samples\n> CER 1.0',
                 transform=ax_zoom.get_yaxis_transform(), va='top',
                 color='red', fontsize=9)
    ax_zoom.set_xlabel('Character Error Rate (CER)')
    ax_zoom.set_ylabel('Anzahl Samples')
    ax_zoom.set_title(f'{label}\nZoom: 0 – 2.0')
    ax_zoom.set_xlim(0, 2)
    ax_zoom.legend(fontsize=9)

    print(f'{label}  (Min={cer_min:.3f}, Max={cer_max:.3f}):')
    for p in [50, 75, 90, 95, 99, 100]:
        print(f'  P{p:>3} = {cer_vals.quantile(p/100):.3f}')
    print(f'  Samples mit CER > 1.0: {(cer_vals > 1.0).sum()}')
    print()

plt.suptitle('CER-Verteilung – Voller Bereich vs. Zoom (natürlicher Bruch?)', fontsize=13)
plt.tight_layout()
plt.show()

### Schwellenwert-Analyse: Wo liegt der natürliche Bruch in der CER-Verteilung?

Bevor ein fixer Schwellenwert gesetzt wird: Verteilung anschauen.  
Gibt es einen klaren Gap zwischen «echten» Transkriptionen und Halluzinationen?

In [ ]:
# Stufe 5: Head-to-Head – welcher Pfad kommt ipa_audio näher?
# Referenz: ipa_audio = was tatsächlich gesprochen wurde
# Vergleich: ipa_reference vs. ipa_swiss_whisper – wer ist näher dran?

def cer(ref, hyp):
    import Levenshtein
    if len(ref) == 0:
        return float('nan')
    return Levenshtein.distance(ref, hyp) / len(ref)

cer_ref   = df.apply(lambda r: cer(r['ipa_audio'], r['ipa_reference']), axis=1)
cer_swiss = df.apply(lambda r: cer(r['ipa_audio'], r['ipa_swiss_whisper']), axis=1)

valid = cer_ref.notna() & cer_swiss.notna()
x = cer_ref[valid]    # CER: ipa_reference  vs. ipa_audio
y = cer_swiss[valid]  # CER: ipa_swiss_whisper vs. ipa_audio

ref_wins   = (x < y).sum()   # reference näher an ipa_audio
swiss_wins = (y < x).sum()   # swiss_whisper näher an ipa_audio
ties       = (x == y).sum()
total      = valid.sum()

colors = pd.Series('gray', index=x.index)
colors[x < y] = '#4C72B0'   # ipa_reference besser
colors[y < x] = '#55A868'   # swiss_whisper besser

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# --- Scatter-Plot ---
ax = axes[0]
ax.scatter(x, y, c=colors, alpha=0.35, s=18, linewidths=0)

lim = max(x.max(), y.max()) * 1.05
ax.plot([0, lim], [0, lim], 'k--', linewidth=1.2, label='Gleichstand')

# Beschriftung der Hälften
ax.text(0.04, 0.92,
        f'Swiss Whisper näher\n({swiss_wins} Samples, {swiss_wins/total:.0%})',
        transform=ax.transAxes, color='#55A868', fontsize=10, va='top')
ax.text(0.58, 0.12,
        f'ipa_reference näher\n({ref_wins} Samples, {ref_wins/total:.0%})',
        transform=ax.transAxes, color='#4C72B0', fontsize=10, va='bottom')

ax.set_xlabel('CER  ipa_reference  vs. ipa_audio  (x-Achse)')
ax.set_ylabel('CER  ipa_swiss_whisper  vs. ipa_audio  (y-Achse)')
ax.set_title('Head-to-Head: Welcher Pfad kommt dem tatsächlich Gesprochenen (ipa_audio) näher?\n'
             'Punkte über Diagonale → Swiss Whisper weiter weg  |  Punkte darunter → Swiss Whisper näher')
ax.set_xlim(0, lim)
ax.set_ylim(0, lim)
ax.set_aspect('equal')
ax.legend(fontsize=9)

# --- Balkendiagramm ---
ax2 = axes[1]
counts     = [ref_wins, swiss_wins, ties]
labels     = ['ipa_reference\nnäher', 'Swiss Whisper\nnäher', 'Gleichstand']
bar_colors = ['#4C72B0', '#55A868', 'lightgray']
bars = ax2.bar(labels, counts, color=bar_colors, edgecolor='white', width=0.5)
for bar, count in zip(bars, counts):
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 15,
             f'{count}\n({count/total:.0%})', ha='center', va='bottom', fontsize=11)
ax2.set_ylabel('Anzahl Samples')
ax2.set_title('Siegesverteilung (vs. ipa_audio)')
ax2.set_ylim(0, max(counts) * 1.2)

plt.tight_layout()
plt.show()

print(f'\nGesamturteil ({total} Samples) – gemessen an ipa_audio:')
print(f'  ipa_reference näher an ipa_audio:      {ref_wins:>5}  ({ref_wins/total:.1%})')
print(f'  Swiss Whisper näher an ipa_audio:      {swiss_wins:>5}  ({swiss_wins/total:.1%})')
print(f'  Gleichstand:                           {ties:>5}  ({ties/total:.1%})')
print()
print(f'  Ø CER ipa_reference vs. ipa_audio:    {x.mean():.3f}')
print(f'  Ø CER swiss_whisper vs. ipa_audio:    {y.mean():.3f}')
delta = y.mean() - x.mean()
winner = "ipa_reference" if delta > 0 else "Swiss Whisper"
print(f'  → {winner} ist im Schnitt näher an ipa_audio (ΔCER = {abs(delta):.3f})')


## Stufe 6: Repetitions-Erkennung in ipa_audio

Halluzinationen des Modells zeigen sich oft als Wiederholungen desselben Musters.  
`has_repetition(text)` erkennt Muster von 2–6 Zeichen, die mindestens 4× hintereinander vorkommen.

In [ ]:
import re

def has_repetition(text: str, min_len: int = 2, max_len: int = 6, min_reps: int = 4) -> bool:
    """
    Gibt True zurück, wenn ein Muster von min_len–max_len Zeichen
    mindestens min_reps-mal hintereinander im Text vorkommt.
    """
    if not isinstance(text, str) or len(text) == 0:
        return False
    for n in range(min_len, max_len + 1):
        # Muster: irgendeine Zeichengruppe der Länge n, (min_reps-1)-mal gefolgt von sich selbst
        pattern = r'(.{' + str(n) + r'})\1{' + str(min_reps - 1) + r',}'
        if re.search(pattern, text):
            return True
    return False

# Auf ipa_audio anwenden
df['_has_rep'] = df['ipa_audio'].apply(has_repetition)

n_rep = df['_has_rep'].sum()
print(f'Samples mit Repetitionen in ipa_audio: {n_rep} von {len(df)} ({n_rep/len(df):.1%})')

# Beispiele
print('\n--- 5 Beispiele mit Repetitionen ---')
for _, row in df[df['_has_rep']].head(5).iterrows():
    print(f'  ipa_audio: {row["ipa_audio"][:120]}')

### Übersicht: Repetitionen vs. CER-Filter

In [ ]:
# Voraussetzung: _cer_ipa_audio muss existieren (aus Stufe 4)
# Falls nicht vorhanden, hier neu berechnen:
import Levenshtein as _lev
if '_cer_ipa_audio' not in df.columns:
    df['_cer_ipa_audio'] = df.apply(
        lambda r: _lev.distance(r['ipa_reference'], r['ipa_audio']) / max(len(r['ipa_reference']), 1),
        axis=1
    )

# Flags
rep      = df['_has_rep']                  # hat Repetition
cer_flag = df['_cer_ipa_audio'] > 1.0      # bereits durch CER > 1 gefiltert

n_total   = len(df)
n_rep     = rep.sum()
n_rep_cer = (rep & cer_flag).sum()          # Repetition UND CER > 1
n_rep_new = (rep & ~cer_flag).sum()         # Repetition aber CER <= 1 (NEU)

print('=== Repetitions-Übersicht ===')
print(f'Gesamt Samples:                  {n_total:>6}')
print(f'Mit Repetition:                  {n_rep:>6}  ({n_rep/n_total:.1%})')
print(f'  → davon CER > 1 (bekannt):     {n_rep_cer:>6}  ({n_rep_cer/n_rep:.1%} der Reps)')
print(f'  → davon NEU (CER <= 1):         {n_rep_new:>6}  ({n_rep_new/n_rep:.1%} der Reps)')

print()
print('---  Beispiele der NEU gefundenen Fälle (Repetition, CER <= 1) ---')
neu = df[rep & ~cer_flag][['_cer_ipa_audio', 'ipa_audio']].head(10)
for _, row in neu.iterrows():
    print(f'  CER={row["_cer_ipa_audio"]:.3f} | {row["ipa_audio"][:100]}')
